In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 12.0 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -q rwkv==0.8.31

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 7.7 MB/s eta 0:00:00a 0:00:01


In [10]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.2.0.dev10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.2/426.2 kB 8.1 MB/s eta 0:00:00a 0:00:01


In [5]:
!pip install -q rwkv huggingface_hub

In [6]:
import os

import huggingface_hub

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

In [12]:
from rwkvx.generation import (
    RWKVSession,
    RWKVTokenizer,
    RWKVModel,
    RWKVSampler,
    GenerationConfig
)

from rwkvx.conversation import (
    History,
    ConversationManager,
    Message,
    RWKV7PromptRenderer,
    get_default_rwkv7_prompt_templates,
)

In [13]:
model_title = "rwkv7-g1b-2.9b-20251205-ctx8192"
# model_title = "rwkv7-g0b-7.2b-20251220-ctx8192"
# model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")

rwkv7-g1b-2.9b-20251205-ctx8192.pth:   0%|          | 0.00/5.90G [00:00<?, ?B/s]

In [14]:
config = GenerationConfig(
    max_new_tokens=1000,
    stop_tokens=[0, 261],  # 0, \n\n
)

print(config)

max_new_tokens=1000 temperature=1.0 top_p=0.3 top_k=0 presence_penalty=0.5 frequency_penalty=0.5 decay_penalty=0.996 stop_tokens=[0, 261]


In [15]:
model = RWKVModel(model_path=model_path, strategy='cuda bf16')

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/4aa0981b66af4f727181a147bcea1457ad7c84cb/rwkv7-g1b-2.9b-20251205-ctx8192 (cuda bf16)



In [17]:
tokenizer = RWKVTokenizer()
sampler = RWKVSampler(config)

In [18]:
templates = get_default_rwkv7_prompt_templates()
renderer = RWKV7PromptRenderer(templates)

mgr = ConversationManager(renderer)

In [19]:
session = RWKVSession(model, tokenizer, sampler, mgr, config)

In [20]:
session.reset()

In [21]:
mgr.add_system('Include emojis to all your answers.')
mgr.build_prompt()

'System: Include emojis to all your answers.'

In [22]:
mgr.add_user('Tell me a short joke.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think'

In [23]:
joke2 = session.generate_message()
print(joke2)

role='assistant' text='\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [24]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾'

In [25]:
mgr.add_user('Tell me a joke different from previous one.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think'

In [26]:
joke = session.generate_message()
print(joke)

role='assistant' text='\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [27]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗'

In [28]:
mgr.add_user('Tell me more complex joke.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think'

In [29]:
joke3 = session.generate_message()
print(joke3)

role='assistant' text='\n😄 Why did the math book look sad? 😟 Because it had too many problems and no solutions! 😂' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [30]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the math book look sad? 😟 Because it had too many problems and no solutions! 😂'

In [31]:
mgr.add_user('List topics of all jokes that you have told me. Explain the second joke.', metadata={'think_mode': ' think'})
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the math book look sad? 😟 Because it had too many problems and no solutions! 😂\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think'

In [32]:
final_answer = session.generate_message()

In [33]:
print(final_answer.metadata['think_text'])

Okay, let's see. The user wants me to list all the topics of jokes I've told them so far and explain the second joke. First, I need to recall all the jokes I've provided. Let me go through each one:
1. The scarecrow winning an award because he was outstanding in his field (corn).
2. The tomato turning red because it saw salad dressing.
3. A math book being sad because of too many problems and no solutions.
4. A math book being sad again, but this time with more complex humor involving a "math book" that's a "book" with math problems and no solutions, leading to sadness due to lack of solutions.
Wait, the user mentioned "more complex joke," so maybe the second one is simpler, and the third one is more complex? Or maybe they want me to explain both as per their request. Let me check again. The user said "Tell me a joke different from previous one." So first joke was about scarecrow, second about tomato, third about math book being sad because of problems and no solutions. Then they asked

In [34]:
print(final_answer.text)

**Jokes Explained:**  
1. **Scarecrow Winning an Award**:  
   *Why did the scarecrow win an award?*  
   Because he was outstanding in his field! 🌾  
   *(Pun: "outstanding" = both "standing out" and "in his field" [corn].)*  
2. **Tomato Turning Red**:  
   *Why did the tomato turn red?*  
   Because it saw salad dressing! 🥗  
   *(Homophone pun: "dressing" sounds like "reddening." The tomato imagines seeing salad dressing (a sauce) and becomes red like it.)*  
3. **Math Book Being Sad**:  
   *Why did the math book look sad?*  
   Because it had too many problems and no solutions! 😂  
   *(Play on words: "problems" = mathematical equations, "solutions" = answers. The book feels overwhelmed by too many problems without answers.)*  
**Explanation of Second Joke**:  
The second joke uses **homophones** (words that sound the same but have different meanings). The phrase "salad dressing" sounds like "reddening," which is a play on words. When you imagine a tomato seeing salad dressing, i

In [35]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the math book look sad? 😟 Because it had too many problems and no solutions! 😂\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>Okay, let\'s see. The user wants me to list all the topics of jokes I\'ve told them so far and explain the second joke. First, I need to recall all the jokes I\'ve provided. Let me go through each one:\n1. The scarecrow winning an award because he was outstanding in his field (corn).\n2. The tomato turning red because it saw salad dressing.\n3. A math book being sad because of too m

In [36]:
mgr.add_user('Lets wrap up. Thank you!')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the math book look sad? 😟 Because it had too many problems and no solutions! 😂\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>Okay, let\'s see. The user wants me to list all the topics of jokes I\'ve told them so far and explain the second joke. First, I need to recall all the jokes I\'ve provided. Let me go through each one:\n1. The scarecrow winning an award because he was outstanding in his field (corn).\n2. The tomato turning red because it saw salad dressing.\n3. A math book being sad because of too m

In [37]:
session.generate_message()

Message(role='assistant', text="\nYou're welcome! It was fun exploring all those jokes with you. If you ever want more laughs or need help with anything else, just let me know! 😊", metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'})